<a href="https://colab.research.google.com/github/vbrasila/agentes-2026-2-equipe/blob/main/enc04_hipoteses_e_evidencia_Nayara.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 4 — O canal nativo de ferramentas

Tópicos Especiais em IA — Agentes Inteligentes · IFES Serra · 2026/2

---

## O que mudou desde a aula passada, e por quê

No Encontro 3 o agente conversava em **texto livre**: o modelo escrevia `ACAO: ...`, e um `regex` extraía o pedido. Isso **parou de funcionar**. Não porque erramos o `regex` — porque o modelo recusa o canal.

A execução real em **24/08/2026**, devolveu isto:

```
BadRequestError: 400 — 'Tool choice is none, but model called a tool'
code: 'tool_use_failed'
failed_generation: '{"name": "commentary", "arguments": We need to produce
the chain of turns. The user asked a question... But we must produce them.'
```

Leia o `failed_generation` com atenção: o modelo **tentou emitir um pedido de ferramenta pelo canal nativo**, e o servidor recusou porque nenhuma ferramenta havia sido declarada. O `gpt-oss` foi treinado com chamada de ferramenta embutida no formato de resposta. Pedir a ele um rastro em texto é pedir que ele use a porta errada.

> **A aula de hoje troca o canal.** O laço é o mesmo do Encontro 3 — o que muda é **como o pedido atravessa a fronteira**.

## O que você entrega ao final desta aula

1. A **linha de base** medida em quatro indicadores, no canal nativo, antes de mexer em nada
2. A **sua hipótese** aplicada e medida **duas vezes**, com a variação entre medições ao lado do efeito
3. O **conflito** entre saída garantida e uso de ferramenta, provocado de propósito e com a mensagem de erro na mão
4. O **custo por resposta aceitável**, em dólares, dos dois caminhos que sobram

## O desenho do experimento

Uma variável por vez. Mesma pergunta, mesmas funções Python, mesmo modelo — muda **uma** coisa, e a gente mede.

| Indicador | O que pergunta |
|---|---|
| **conclusão** | o agente chegou a uma resposta dentro do orçamento de voltas? |
| **chamadas mal aproveitadas** | quantas chamadas voltaram erro, câmara inexistente ou lote sem ficha? |
| **evidência** (0 a 3) | a resposta final cita a leitura, a faixa e o tempo fora? |
| **custo** | quanto custou, em dólares |

> **O segundo indicador é novo, e ele substitui a "obediência" do Encontro 3.** Obediência de formato deixou de ser mensurável: o protocolo **garante** a estrutura, ou devolve `400`. O que sobra para medir é se o modelo escolheu **a ferramenta certa com o argumento certo** — e isso depende do texto das descrições.

## Como este notebook é usado

**Ele está completo.** Nada a preencher. Vocês **rodam junto com o professor**, célula por célula, e a sua parte é **escolher, medir e ler o resultado**.

**Caminho estendido** no fim, para quem terminar antes.

## Parte 0 — Célula de preparo

A máquina do Colab é apagada entre sessões. Toda aula começa aqui.

In [1]:
%pip install -q "openai>=1.99.0,<3"

import importlib.metadata as md
print("openai", md.version("openai"))

openai 2.54.0


## Parte 1 — Chave, modelos e **o preço**

Três novidades em relação ao Encontro 3.

**Primeira: os modelos mudaram.** O `llama-3.1-8b-instant` e o `llama-3.3-70b-versatile` não estavam mais no catálogo do Groq quando conferimos, em **21/08/2026**. No lugar entram os dois `gpt-oss`, que são **modelos de raciocínio**: raciocinam num campo próprio da resposta, e **os tokens de raciocínio são cobrados como saída**.

**Segunda: agora temos o preço.** E ele é assimétrico:

| Modelo | Entrada / 1M | Saída / 1M |
|---|---|---|
| `openai/gpt-oss-20b` | US$ 0,075 | **US$ 0,30** |
| `openai/gpt-oss-120b` | US$ 0,15 | **US$ 0,60** |

**Saída custa 4× a entrada.** Numa execução medida no Encontro 3, a saída foi **27% dos tokens e 60% do custo**.

**Terceira, e é a que muda a aritmética de hoje: no canal nativo a entrada cresceu.** A declaração de `tools` vai no *prompt* de **toda** volta, e o número de voltas subiu. A estimativa é **~3.300 tokens por execução**, contra ~2.000 no Encontro 3.

### E os limites do plano gratuito, que agora apertam

| RPM | RPD | **TPM** | TPD |
|---|---|---|---|
| 30 | 1.000 | **8.000** | 200.000 |

| Ação | Tokens *(est.)* | % do TPM |
|---|---|---|
| uma execução | ~3.300 | **41%** |
| uma medição de **duas** execuções | ~6.600 | **83%** |
| três execuções no mesmo minuto | ~9.900 | **estoura** |

> **Regra do laboratório de hoje: uma medição por minuto, com `n = 2`.** As células já têm o `time.sleep(60)` onde ele é obrigatório. Se você disparar duas seguidas, vai pegar `429`: o código espera e repete sozinho, e a medição **continua válida** — você só perde tempo.

> **Uma alavanca que existe e que hoje nós não usamos.** Os dois `gpt-oss` aceitam o parâmetro `reasoning_effort` com valores `low`, `medium` e `high`, que controla **quantos tokens de raciocínio** o modelo gasta. Como raciocínio é cobrado como saída, e saída é ~60% do custo, esse é **o parâmetro mais direto sobre a conta desta aula** — e ele fica de fora do experimento de propósito, para não virar uma segunda variável. **É o primeiro item do caminho estendido**, e é o de maior retorno.

In [2]:
import os, time, json, re, unicodedata
from openai import OpenAI

def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor

LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_API_KEY = obter_chave("GROQ_API_KEY")

MODELOS_CANDIDATOS = ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]
LLM_MODEL = MODELOS_CANDIDATOS[0]

# Preco publicado em console.groq.com/pricing, conferido em 21/08/2026.
# Dolares por MILHAO de tokens.
PRECOS = {
    "openai/gpt-oss-20b":  {"entrada": 0.075, "saida": 0.30},
    "openai/gpt-oss-120b": {"entrada": 0.150, "saida": 0.60},
}

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print("chave carregada, termina em:", LLM_API_KEY[-4:])
print("modelo:", LLM_MODEL)

chave carregada, termina em: ZEJj
modelo: openai/gpt-oss-20b


### A chamada que tolera falha

Igual à do Encontro 3, com um acréscimo: **ela lê o cabeçalho `retry-after`** quando o provedor manda. O Groq devolve esse cabeçalho no `429`, em segundos — esperar o que ele pede é melhor do que adivinhar.

In [3]:
def _espera_sugerida(erro, padrao: float) -> float:
    """Le o cabecalho retry-after, se o provedor mandou. Senao usa o padrao."""
    try:
        cab = getattr(getattr(erro, "response", None), "headers", {}) or {}
        v = cab.get("retry-after") or cab.get("Retry-After")
        if v:
            return float(v)
    except (TypeError, ValueError):
        pass
    return padrao


def chamar(mensagens, temperatura: float = 0.0, tentativas: int = 5,
           max_tokens: int = 2000, **extra):
    """Chama o modelo tolerando 429/503, com espera crescente e troca de modelo.

    IMPORTANTE: gpt-oss e modelo de raciocinio. Os tokens de raciocinio
    consomem o orcamento de SAIDA antes do texto final. Com max_tokens curto,
    o raciocinio ocupa tudo e a resposta volta VAZIA ou cortada — sem erro.
    Por isso 2000, e por isso avisamos quando finish_reason == "length".

    Args:
        mensagens: lista de mensagens no formato da API
        temperatura: 0.0 para reproduzir, 1.0 para medir variacao
        max_tokens: teto de saida. Nao baixe sem medir.
        extra: repassado a API — usado para response_format na Parte 8
    """
    espera = 2.0
    for t in range(tentativas):
        try:
            r = cliente.chat.completions.create(
                model=LLM_MODEL, messages=mensagens,
                temperature=temperatura, max_tokens=max_tokens, **extra
            )
            if r.choices[0].finish_reason == "length":
                print("  [AVISO: resposta CORTADA por max_tokens. O texto final "
                      "pode estar incompleto ou vazio. Aumente max_tokens.]")
            return r
        except Exception as e:
            codigo = getattr(e, "status_code", None)
            if codigo not in (429, 500, 502, 503, 504) or t == tentativas - 1:
                raise
            pausa = _espera_sugerida(e, espera)
            print(f"  [{codigo}] esperando {pausa:.0f}s  (nao invalida a medicao)")
            time.sleep(pausa)
            espera *= 2
    raise RuntimeError("todas as tentativas falharam")


# NOTA DE PROJETO, e ela vale ler em voz alta.
#
# O notebook do Encontro 3 trocava de modelo depois de tres 429. AQUI ISSO FOI
# REMOVIDO, de proposito. Motivo: hoje o modelo E UMA VARIAVEL DO EXPERIMENTO.
# Uma troca automatica no meio de uma medicao faria tres coisas ruins e todas
# em silencio:
#   1. metade da amostra viria de um modelo e metade de outro;
#   2. a equipe da hipotese H4 — que testa exatamente a troca de modelo —
#      seria rebaixada justamente na variavel que ela mede;
#   3. custo_usd() aplicaria o preco do modelo FINAL a tokens gerados pelo
#      outro, e a conta em dolares do bloco D sairia errada sem avisar.
#
# A licao: tolerancia a falha e boa em producao e PERIGOSA em medicao. Um
# sistema que se conserta sozinho esconde a variavel que voce esta estudando.


def custo_usd(entrada: int, saida: int, modelo: str = None) -> float:  # noqa: E302
    """Custo em dolares de uma chamada, pelo preco publicado do Groq.

    Args:
        entrada: tokens de prompt
        saida: tokens de completion, incluindo os de raciocinio
        modelo: id do modelo. Se omitido, usa o LLM_MODEL corrente.
    """
    p = PRECOS[modelo or LLM_MODEL]
    return (entrada * p["entrada"] + saida * p["saida"]) / 1_000_000

print("chamada e calculo de custo prontos.")

chamada e calculo de custo prontos.


## Parte 2 — As funções, iguais às do Encontro 3

**As funções Python não mudam uma linha, e isso é de propósito.** O que muda hoje é o **canal**, não o domínio nem a lógica. Se as ferramentas mudassem, não daria para atribuir nada ao canal.

Dados sintéticos, mesmo domínio: câmaras frias, lotes de insumo, excursões de temperatura.

> Repare, ao ler, que as *docstrings* continuam aqui. Elas vão ser **traduzidas** na próxima parte — e a tradução é o assunto da aula.

In [4]:
def temperatura_camara(camara: str) -> str:
    """Le a temperatura atual de uma camara fria e devolve o valor em graus Celsius.

    Use quando precisar da condicao ATUAL de uma camara. Nao devolve historico:
    para saber se ela saiu da faixa antes, use historico_excursoes.

    Args:
        camara: identificador da camara, no formato CF-NN. Exemplo: "CF-02".
    """
    leituras = {"CF-01": 4.2, "CF-02": 9.8, "CF-03": -21.5}
    if camara not in leituras:
        return f"camara {camara} desconhecida"
    return f"{camara}: {leituras[camara]} graus Celsius neste momento"


def especificacao_do_insumo(lote: str) -> str:
    """Devolve em que camara um lote esta guardado, a faixa de temperatura
    permitida para ele, e a data de validade.

    Args:
        lote: identificador do lote, no formato L-NN. Exemplo: "L-77".
    """
    fichas = {
        "L-77": ("reagente enzimatico; guardado na camara CF-02; "
                 "faixa permitida de 2 a 8 C; validade 2026-11-30"),
        "L-88": ("meio de cultura; guardado na camara CF-01; "
                 "faixa permitida de 2 a 8 C; validade 2026-09-15"),
        "L-91": ("enzima de restricao; guardado na camara CF-03; "
                 "faixa permitida de -25 a -15 C; validade 2027-02-28"),
    }
    return fichas.get(lote, f"sem especificacao para o lote {lote}")


def historico_excursoes(camara: str, horas: str = "24") -> str:
    """Lista as excursoes de temperatura de uma camara nas ultimas N horas.
    Uma excursao e um periodo em que a camara saiu da faixa permitida.

    Args:
        camara: identificador da camara, no formato CF-NN. Exemplo: "CF-02".
        horas: janela em horas, como texto. Exemplo: "24".
    """
    base = {
        "CF-01": [],
        "CF-02": [("-3h", "subiu a 9,8 C e ainda nao voltou"),
                  ("-19h", "pico de 8,6 C por cerca de 40 min")],
        "CF-03": [("-30h", "queda a -28 C por cerca de 15 min")],
    }
    if camara not in base:
        return f"camara {camara} desconhecida"
    try:
        janela = int(float(horas))
    except (TypeError, ValueError):
        return "ERRO: horas deve ser um numero, por exemplo 24"
    dentro = [f"{q} {d}" for q, d in base[camara] if int(q.strip("-h")) <= janela]
    if not dentro:
        return f"{camara}: 0 excursoes nas ultimas {janela}h"
    return (f"{camara}: {len(dentro)} excursao(oes) nas ultimas {janela}h -- "
            + "; ".join(dentro))


FERRAMENTAS = {
    "temperatura_camara": temperatura_camara,
    "especificacao_do_insumo": especificacao_do_insumo,
    "historico_excursoes": historico_excursoes,
}

PERGUNTA = "O lote L-77 ainda pode ser usado?"
print("ferramentas:", list(FERRAMENTAS))
print("pergunta fixa do experimento:", PERGUNTA)

ferramentas: ['temperatura_camara', 'especificacao_do_insumo', 'historico_excursoes']
pergunta fixa do experimento: O lote L-77 ainda pode ser usado?


## Parte 3 — A mesma informação, agora em JSON

No Encontro 3, o que o modelo sabia sobre as ferramentas estava **dentro da instrução**, escrito à mão:

```
Ferramentas disponiveis:
- especificacao_do_insumo("L-77") -> em que camara o lote esta, a faixa...
```

Agora essa informação vai **num campo próprio da requisição**, o `tools`. Três consequências, e as três importam:

| Consequência | Por quê |
|---|---|
| a instrução **encolhe** | a lista de ferramentas saiu dela |
| a *docstring* virou **`description`** | é o campo que o modelo lê para escolher |
| o argumento virou **esquema com tipo** | `{"type": "string"}` — o protocolo passa a saber a forma |

**E aqui está o ponto da clínica de hoje, em uma frase:** o campo `description` é o único lugar onde você fala com o modelo sobre a ferramenta. Se ele estiver vazio de informação, o modelo escolhe às cegas — e nós vamos **medir** isso.

Por isso a célula abaixo declara **duas** versões: a rica e a pobre. A pobre é a hipótese **H2**.

### O formalismo do `parameters`, em cinco linhas

Você vai ter de **escrever** isto na tarefa de casa, então vale ler devagar. O `parameters` é um **JSON Schema**, o mesmo formalismo que a Parte 9 usa para garantir a saída:

| Chave | O que diz | No nosso caso |
|---|---|---|
| `"type": "object"` | os argumentos vêm como **um objeto**, com nomes | sempre isto |
| `"properties"` | **um item por argumento**, com `type` e `description` | `lote`, `camara`, `horas` |
| `"type": "string"` | o tipo daquele argumento | tudo texto, aqui |
| `"description"` | o que aquele argumento é — **e é onde vai o exemplo** | *"formato L-NN. Exemplo: L-77"* |
| `"required"` | a **lista** dos argumentos obrigatórios | `["lote"]` |

**A linha que mais rende, e é a linha 3 do critério da clínica:** `"type": "string"` **não diz nada sobre formato**. `"CF-02"` e `"câmara 2"` são as duas *strings* válidas. Só o **exemplo** dentro do `description` desempata.

> **Um detalhe do vocabulário, que responde à mensagem de erro da abertura.** Existe um parâmetro irmão do `tools` chamado **`tool_choice`**, que diz se o modelo **pode**, **deve** ou **não pode** chamar ferramenta. Quando você não declara `tools`, ele vale `"none"` — *"não pode"*. Foi esse `"none"` que o servidor citou ao recusar: *"Tool choice is none, but model called a tool"*. A mensagem estava correta; era o nosso desenho que estava errado.

In [5]:
# A MESMA informacao das docstrings, agora no formato que o protocolo carrega.
TOOLS = [
    {"type": "function", "function": {
        "name": "especificacao_do_insumo",
        "description": (
            "Devolve a ficha de um LOTE: em que camara ele esta guardado, a faixa "
            "de temperatura permitida para ele, e a data de validade. Nao devolve "
            "leitura de temperatura: para isso use temperatura_camara."),
        "parameters": {
            "type": "object",
            "properties": {
                "lote": {"type": "string",
                         "description": "identificador do lote, no formato L-NN. Exemplo: L-77"},
            },
            "required": ["lote"],
        },
    }},
    {"type": "function", "function": {
        "name": "temperatura_camara",
        "description": (
            "Le a temperatura ATUAL de uma camara fria e devolve o valor em graus "
            "Celsius. Nao devolve historico: para saber se a camara saiu da faixa "
            "antes, use historico_excursoes."),
        "parameters": {
            "type": "object",
            "properties": {
                "camara": {"type": "string",
                           "description": "identificador da camara, no formato CF-NN. Exemplo: CF-02"},
            },
            "required": ["camara"],
        },
    }},
    {"type": "function", "function": {
        "name": "historico_excursoes",
        "description": (
            "Lista as excursoes de temperatura de uma camara nas ultimas N horas. "
            "Uma excursao e um periodo em que a camara saiu da faixa permitida. "
            "Devolve historico, nao a condicao atual: para a leitura de agora use "
            "temperatura_camara."),
        "parameters": {
            "type": "object",
            "properties": {
                "camara": {"type": "string",
                           "description": "identificador da camara, no formato CF-NN. Exemplo: CF-02"},
                "horas": {"type": "string",
                          "description": "janela em horas, como texto. Exemplo: 24"},
            },
            "required": ["camara"],
        },
    }},
]

# INVARIANTE DO EXPERIMENTO, e e o que faz o par H2 x H3 valer:
# as descricoes acima dizem apenas ESCOPO — o que a ferramenta devolve, o que
# ela NAO cobre, e o formato do argumento com exemplo. Elas NAO dizem em que
# ORDEM usar as ferramentas. Essa regra vive so na instrucao, e e o que a H3
# apaga. Se voce escrever "use esta primeiro" numa description, as duas
# hipoteses passam a mexer na mesma coisa e o par deixa de isolar nada.

# A variante POBRE, para a hipotese H2. MESMOS nomes, MESMOS argumentos —
# so as descricoes foram esvaziadas, cada uma no estilo de um especime da
# clinica: a primeira serve para qualquer pergunta (especime 2), a segunda
# descreve o trabalho interno da funcao (especime 1), a terceira usa
# vocabulario que so quem escreveu entende (especime 4). E as tres perderam
# o EXEMPLO do argumento (especime 3).
TOOLS_POBRE = [
    {"type": "function", "function": {
        "name": "especificacao_do_insumo",
        "description": "Retorna informacoes sobre o item solicitado.",
        "parameters": {"type": "object", "required": ["lote"], "properties": {
            "lote": {"type": "string", "description": "o identificador"}}}}},
    {"type": "function", "function": {
        "name": "temperatura_camara",
        "description": "Faz uma consulta no sistema e processa o resultado.",
        "parameters": {"type": "object", "required": ["camara"], "properties": {
            "camara": {"type": "string", "description": "o identificador"}}}}},
    {"type": "function", "function": {
        "name": "historico_excursoes",
        "description": "Consulta o historico conforme o fluxo padrao da unidade.",
        "parameters": {"type": "object", "required": ["camara"], "properties": {
            "camara": {"type": "string", "description": "o identificador"},
            "horas": {"type": "string", "description": "o periodo"}}}}},
]

# INVARIANTE. Declaracao e implementacao sao DUAS fontes de verdade, e elas
# podem divergir em silencio: basta renomear a funcao e esquecer o JSON.
# Quando divergem, o modelo pede uma ferramenta que nao existe.
_declaradas = {t["function"]["name"] for t in TOOLS}
assert _declaradas == set(FERRAMENTAS), \
    f"declaracao e registro divergem: {_declaradas ^ set(FERRAMENTAS)}"
assert {t["function"]["name"] for t in TOOLS_POBRE} == _declaradas, \
    "a variante pobre tem de declarar as MESMAS ferramentas"

print("declaradas e implementadas:", sorted(_declaradas))
print()
print(f"declaracao rica : {len(json.dumps(TOOLS)):5d} caracteres")
print(f"declaracao pobre: {len(json.dumps(TOOLS_POBRE)):5d} caracteres")
print()
print("Guarde os dois numeros. A declaracao inteira e reenviada em TODA volta")
print("do laco: a diferenca acima e o preco da descricao boa, por chamada.")
print()
# Conferencia do invariante: nenhuma description pode dar ordem de uso.
_proibido = ["sempre primeiro", "primeiro quando", "comece pela", "depois use"]
for _t in TOOLS:
    _d = _t["function"]["description"].lower()
    assert not any(p in _d for p in _proibido), (
        f"a description de {_t['function']['name']} da ORDEM DE USO. "
        "Isso pertence a instrucao, senao a H2 e a H3 mexem na mesma coisa.")
print("invariante ok: as descriptions dizem escopo, nao ordem de uso.")

declaradas e implementadas: ['especificacao_do_insumo', 'historico_excursoes', 'temperatura_camara']

declaracao rica :  1469 caracteres
declaracao pobre:   879 caracteres

Guarde os dois numeros. A declaracao inteira e reenviada em TODA volta
do laco: a diferenca acima e o preco da descricao boa, por chamada.

invariante ok: as descriptions dizem escopo, nao ordem de uso.


## Parte 4 — A instrução, e o laço pelo canal nativo

Compare as duas instruções da célula abaixo. A do Encontro 3 está lá para contraste, e **o que desapareceu é o conteúdo desta parte**:

| Saiu da instrução | Para onde foi |
|---|---|
| o bloco `PENSAMENTO: / ACAO: / RESPOSTA:` | **para o protocolo** — o pedido chega em campo próprio |
| a lista de ferramentas | para o parâmetro `tools` |
| *"UMA ação por volta. Nunca duas."* | **para lugar nenhum, e isso é uma perda** — veja abaixo |

**A regra que desapareceu merece parágrafo.** No canal nativo o modelo **pode** pedir várias ferramentas na mesma volta, e isso é legítimo: se ele precisa de dois dados independentes, pedir os dois de uma vez economiza uma volta inteira. Nós executamos todos. Então *"uma ação por volta"* deixou de ser regra — e a "obediência" que vocês mediram no Encontro 3 deixou de existir como indicador.

**O que sobrou na instrução são só as duas regras de domínio:** comece pela especificação, e verifique o histórico antes de concluir. São elas o alvo da hipótese **H3**.

### As três linhas em que o laço mudou

```python
if not msg.tool_calls:        # 1. resposta final é DEFINIÇÃO, não leitura de texto
    return ...
mensagens.append({"role": "assistant", "tool_calls": [...]})   # 2. o turno dele volta com os pedidos
mensagens.append({"role": "tool", "tool_call_id": tc.id, ...}) # 3. o quarto papel, enfim
```

> **O papel `tool`.** Nas aulas anteriores nós empurrávamos a observação como se fosse fala do usuário — `{"role": "user", "content": "OBSERVACAO: ..."}` — e eu disse na ocasião que o protocolo tem um papel próprio para isso. É este. Ele carrega `tool_call_id`, que **amarra a resposta ao pedido**: com três chamadas na mesma volta, é o `id` que diz qual resultado é de qual.

In [6]:
# ---------------------------------------------------------------------
# A instrucao do Encontro 3, guardada SO para contraste e para a
# demonstracao do erro. Nao e usada no experimento.
# ---------------------------------------------------------------------
INSTRUCAO_ENC3 = """Voce e um agente que avalia se insumos refrigerados podem ser usados.

Trabalhe em voltas. Em CADA volta escreva exatamente uma destas duas formas:

PENSAMENTO: <seu raciocinio em uma linha>
ACAO: <nome_da_ferramenta>("<argumento>")

ou, quando ja tiver todos os dados de que precisa:

PENSAMENTO: <seu raciocinio em uma linha>
RESPOSTA: <resposta final ao responsavel pelo almoxarifado>

Ferramentas disponiveis:
- especificacao_do_insumo("L-77") -> em que camara o lote esta, a faixa permitida e a validade
- temperatura_camara("CF-02") -> temperatura atual daquela camara
- historico_excursoes("CF-02", "24") -> quando a camara saiu da faixa nas ultimas 24h

Regras:
- UMA acao por volta. Nunca duas.
- Nunca invente uma leitura nem uma faixa. Se precisa de um dado, use a ferramenta.
- Para decidir sobre um LOTE, comece pela especificacao dele: e ela que diz
  em qual camara ele esta e qual faixa vale. Sem isso voce nao sabe o que medir.
- Se a temperatura estiver fora da faixa, verifique o historico de excursoes
  antes de concluir: o tempo fora da faixa muda a conclusao.
"""
# Conferencia: e a instrucao do Encontro 3, sem uma virgula alterada.
# Se este numero mudar, a comparacao de tamanho do slide fica errada.
assert len(INSTRUCAO_ENC3) == 1083, (
    f"INSTRUCAO_ENC3 tem {len(INSTRUCAO_ENC3)} car., esperado 1083 — "
    "ela precisa ser IDENTICA a do Encontro 3 para a comparacao valer")

# ---------------------------------------------------------------------
# A instrucao BASE de hoje. E o "antes" do experimento.
# ---------------------------------------------------------------------
INSTRUCAO_BASE = """Voce e um agente que avalia se insumos refrigerados podem ser usados.

Use as ferramentas disponiveis para obter os dados. Nunca invente uma leitura
nem uma faixa de temperatura: se precisa de um dado, chame a ferramenta.

Para decidir sobre um LOTE, comece pela especificacao dele: e ela que diz em
qual camara ele esta e qual faixa vale. Sem isso voce nao sabe o que medir.

Se a temperatura estiver fora da faixa, verifique o historico de excursoes
antes de concluir: o tempo fora da faixa muda a conclusao.

Quando tiver todos os dados, responda em texto ao responsavel pelo almoxarifado.
"""

# A ultima linha da instrucao, isolada porque tres variantes a substituem.
_FECHO = ("Quando tiver todos os dados, responda em texto ao responsavel "
          "pelo almoxarifado.")
assert _FECHO in INSTRUCAO_BASE, "o fecho tem de existir na instrucao base"


def executar(nome: str, argumentos_json: str) -> str:
    """Roda a ferramenta pedida pelo modelo. Devolve SEMPRE texto, nunca excecao.

    O dicionario FERRAMENTAS e a fronteira de seguranca, e continua sendo a
    mesma do Encontro 3: o modelo manda um NOME, e so um nome que esta no
    dicionario vira chamada. Nada de eval, nada de getattr em modulo.

    A novidade e que os argumentos chegam como TEXTO JSON, nao como lista.
    Isso e melhor: vem com nome de parametro, entao a ordem nao importa mais.
    """
    if nome not in FERRAMENTAS:
        return (f"ERRO: ferramenta '{nome}' nao existe. "
                f"Disponiveis: {', '.join(FERRAMENTAS)}")
    try:
        args = json.loads(argumentos_json or "{}")
    except json.JSONDecodeError as e:
        return f"ERRO: os argumentos nao sao JSON valido: {e}"
    if not isinstance(args, dict):
        return "ERRO: os argumentos precisam ser um objeto JSON"
    try:
        return FERRAMENTAS[nome](**args)
    except TypeError as e:
        return f"ERRO: argumentos invalidos para {nome}: {e}"


def mal_aproveitada(observacao: str) -> bool:
    """A chamada trouxe dado util, ou voltou vazia de informacao?

    Este e o indicador que substituiu a "obediencia". Ele nao mede formato —
    mede ESCOLHA: ferramenta inexistente, argumento fora do formato, camara
    ou lote que nao existe. Tudo isso e falha de descricao, nao de sintaxe.
    """
    o = (observacao or "").lower()
    return (o.startswith("erro")
            or "desconhecida" in o
            or "sem especificacao" in o)


def agente(pergunta: str, instrucao: str, tools=None, max_iteracoes: int = 6,
           verboso: bool = False, **extra):
    """Roda o laco pelo canal NATIVO de ferramentas e devolve os indicadores.

    Args:
        pergunta: o que o responsavel quer saber
        instrucao: a instrucao de sistema
        tools: a declaracao das ferramentas. TOOLS por padrao; TOOLS_POBRE na H2
        max_iteracoes: teto de voltas. Nunca remova: e o unico freio do laco.
        verboso: imprime o rastro
        extra: repassado a API
    """
    tools = TOOLS if tools is None else tools
    mensagens = [{"role": "system", "content": instrucao},
                 {"role": "user", "content": pergunta}]
    entrada = saida = chamadas = ruins = 0

    for volta in range(1, max_iteracoes + 1):
        r = chamar(mensagens, tools=tools, **extra)
        msg = r.choices[0].message
        entrada += r.usage.prompt_tokens
        saida += r.usage.completion_tokens

        if verboso:
            print(f"--- volta {volta} " + "-" * 46)

        # Volta SEM pedido de ferramenta e, por DEFINICAO, a resposta final.
        # Compare com o Encontro 3, onde isso era um regex procurando "RESPOSTA:".
        if not msg.tool_calls:
            texto = msg.content or ""
            if verboso:
                print("RESPOSTA FINAL:", texto.strip()[:1200])
            return {"resposta": texto, "voltas": volta, "entrada": entrada,
                    "saida": saida, "chamadas": chamadas, "ruins": ruins,
                    "concluiu": True}

        # Duas coisas OBRIGATORIAS, e nesta ordem:
        # (1) devolver o turno do proprio modelo, COM os pedidos dentro dele;
        # (2) uma mensagem "tool" por pedido, ligada pelo tool_call_id.
        # Faltando qualquer uma das duas, a proxima chamada da 400.
        mensagens.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        if verboso and len(msg.tool_calls) > 1:
            print(f"  [{len(msg.tool_calls)} chamadas nesta volta — "
                  f"legitimo no canal nativo, e economiza voltas]")

        for tc in msg.tool_calls:
            obs = executar(tc.function.name, tc.function.arguments)
            chamadas += 1
            perdida = mal_aproveitada(obs)
            ruins += perdida
            if verboso:
                marca = "  [MAL APROVEITADA]" if perdida else ""
                print(f"  {tc.function.name}({tc.function.arguments}){marca}")
                print(f"    -> {obs}")
            mensagens.append({"role": "tool", "tool_call_id": tc.id,
                              "name": tc.function.name, "content": obs})

    return {"resposta": f"PAREI POR ORCAMENTO em {max_iteracoes} voltas.",
            "voltas": max_iteracoes, "entrada": entrada, "saida": saida,
            "chamadas": chamadas, "ruins": ruins, "concluiu": False}


print("agente por canal nativo pronto.")
print(f"instrucao do Encontro 3: {len(INSTRUCAO_ENC3):5d} caracteres")
print(f"instrucao de hoje      : {len(INSTRUCAO_BASE):5d} caracteres  "
      f"({100 * (1 - len(INSTRUCAO_BASE) / len(INSTRUCAO_ENC3)):.0f}% menor)")
print(f"declaracao de tools    : {len(json.dumps(TOOLS)):5d} caracteres  "
      f"(nao existia no Encontro 3)")
print(f"SOMA de hoje           : {len(INSTRUCAO_BASE) + len(json.dumps(TOOLS)):5d} caracteres")
print()
print("ATENCAO antes de comemorar a instrucao menor: a soma e MAIOR que a")
print("instrucao do Encontro 3, e as duas parcelas vao no prompt de TODA")
print("volta. O texto nao encolheu — ele MUDOU DE LUGAR e cresceu.")
print("A conta de tokens do Lab 0 e que diz o tamanho do estrago.")

agente por canal nativo pronto.
instrucao do Encontro 3:  1083 caracteres
instrucao de hoje      :   593 caracteres  (45% menor)
declaracao de tools    :  1469 caracteres  (nao existia no Encontro 3)
SOMA de hoje           :  2062 caracteres

ATENCAO antes de comemorar a instrucao menor: a soma e MAIOR que a
instrucao do Encontro 3, e as duas parcelas vao no prompt de TODA
volta. O texto nao encolheu — ele MUDOU DE LUGAR e cresceu.
A conta de tokens do Lab 0 e que diz o tamanho do estrago.


### O erro do canal antigo, reproduzido

Uma chamada, e ela existe para **falhar**. Mandamos a instrução do Encontro 3 **sem declarar `tools`** — exatamente a configuração que quebrou em 24/08.

Duas ressalvas honestas:

1. **A falha é intermitente.** O professor rodou cinco vezes com sucesso antes de ela aparecer. Se a célula passar, isso não desmente nada: significa que nesta tentativa o modelo escolheu o canal de texto. Um agente que funciona *na maioria das vezes* é justamente o problema.
2. **É uma chamada só**, ~500 tokens. Não repita em série.

> **Se der o `400`, leia o `failed_generation` em voz alta.** É a única vez no semestre em que vocês vão ver o modelo tentando falar pela porta errada, e o servidor barrando.

In [7]:
# Esta celula EXISTE PARA FALHAR. O erro e o conteudo.
#
# Reproduz a configuracao EXATA do incidente de 24/08/2026: a instrucao do
# Encontro 3, SEM tools declarado, no modelo em que ele aconteceu (o 120b).
# Por isso trocamos o modelo aqui e devolvemos depois — se rodassemos no 20b
# nao seria o mesmo experimento, e a demonstracao perderia o valor de ser real.
_modelo_guardado = LLM_MODEL
LLM_MODEL = "openai/gpt-oss-20b"
print(f"modelo desta celula: {LLM_MODEL}  (o do incidente)")
print()
try:
    r = chamar([{"role": "system", "content": INSTRUCAO_ENC3},
                {"role": "user", "content": PERGUNTA}])
    print("Nao deu erro nesta tentativa. O modelo escolheu o canal de texto.")
    print("Isso NAO desmente o problema: a falha e intermitente, e agente que")
    print("funciona 'quase sempre' e agente que vai quebrar em producao.")
    print()
    print("=== o que ele escreveu ===")
    print((r.choices[0].message.content or "(vazio)")[:600])
    print()
    print("Se voce ficou sem o erro, use o que esta no slide: ele e a saida")
    print("real desta mesma celula, rodada pelo professor em 24/08/2026.")
except Exception as e:
    print(f"{type(e).__name__}")
    print()
    print(str(e)[:1200])
    print()
    print("--- leia o campo failed_generation acima ---")
    print("O modelo tentou emitir um PEDIDO DE FERRAMENTA pelo canal nativo.")
    print("Como nenhuma ferramenta foi declarada, o servidor recusou.")
    print("Nao ha regex que conserte isso: o texto nunca chegou a existir.")
    print()
    print("E repare na palavra 'Tool choice' na mensagem: tool_choice e um")
    print("parametro da requisicao que diz se o modelo PODE, DEVE ou NAO PODE")
    print("chamar ferramenta. Sem declarar tools, ele vale 'none' — e foi esse")
    print("'none' que o servidor citou ao recusar.")
finally:
    LLM_MODEL = _modelo_guardado
    print()
    print(f"modelo restaurado para: {LLM_MODEL}")

modelo desta celula: openai/gpt-oss-20b  (o do incidente)

BadRequestError

Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "repo_browser.write", "arguments": {"role":"assistant","content":"PENSAMENTO: Need to get specification of L-77.\\nACAO: especificacao_do_insumo(\\"L-77\\")"}}'}}

--- leia o campo failed_generation acima ---
O modelo tentou emitir um PEDIDO DE FERRAMENTA pelo canal nativo.
Como nenhuma ferramenta foi declarada, o servidor recusou.
Nao ha regex que conserte isso: o texto nunca chegou a existir.

E repare na palavra 'Tool choice' na mensagem: tool_choice e um
parametro da requisicao que diz se o modelo PODE, DEVE ou NAO PODE
chamar ferramenta. Sem declarar tools, ele vale 'none' — e foi esse
'none' que o servidor citou ao recusar.

modelo restaurado para: openai/gpt-oss-20b


## Parte 5 — O avaliador de evidência

**Esta é a célula mais importante do semestre até aqui**, e ela é a única parte do laboratório que o canal novo **não** mudou, e ela vem pronta para você ler, não para escrever.

`avaliar_evidencia` lê a resposta final do agente e devolve uma nota de **0 a 3**: quantos dos três fatos ela cita. Os três fatos são os que a resposta *precisaria* trazer para alguém decidir sobre o lote sem confiar no agente:

| Fato | Por que importa |
|---|---|
| a **leitura** (9,8 °C) | é o dado que motiva a decisão |
| a **faixa** (2 a 8 °C) | sem ela, 9,8 não quer dizer nada |
| o **tempo fora** (~3 h) | é o que diferencia "excursão breve" de "lote perdido" |

Repare no que essa função **é**: uma **rubrica escrita em código**. Ela transforma *"a resposta é boa"* — que é opinião — em um número que dá para comparar entre execuções.

> **É por isso que ela é o artefato desta aula.** No Encontro 13 ela cresce e vira suíte de avaliação; a diferença entre lá e aqui é quantidade, não natureza.

E repare no que ela **não** é: ela não checa se a resposta está **correta**. Ela checa se a resposta é **auditável**. Um agente pode citar 9,8 e a faixa 2 a 8 e concluir errado — e o avaliador dá 3. Guardem essa limitação: ela é o assunto do Encontro 13.

In [8]:
def _normalizar(s: str) -> str:
    """Minusculas, sem acento, virgula decimal como ponto — para o regex nao errar."""
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace(",", ".")


# Os padroes sao DELIBERADAMENTE tolerantes a formatacao. Motivo: a hipotese
# H1 pede ao modelo texto mais elaborado, entao ela e a mais exposta a perder
# ponto por um travessao em vez de um "a". Falso negativo aqui seria
# indistinguivel de "H1 nao funcionou" — e estragaria o experimento central.
UNIDADE = r"(?:\s*°?\s*(?:c|celsius|graus?)\b)?"   # opcional: °C, C, graus, celsius
SEPARADOR = r"(?:a|ate|e|-|--|–|—|/|to)"           # "2 a 8", "2-8", "2 to 8"

FATOS = {
    # a leitura: 9.8 em qualquer notacao (a normalizacao ja trocou , por .)
    "leitura": [r"9\.8"],
    # a faixa: os tres padroes sao redundantes de proposito. O terceiro e
    # rede de seguranca: dois digitos proximos, sem cruzar outro digito
    # nem quebra de linha.
    "faixa": [
        rf"\b2{UNIDADE}\s*{SEPARADOR}\s*8\b",
        r"entre\s+2\b[^\d\n]{0,14}?8\b",
        r"\b2\b[^\d\n]{0,14}?\b8\b",
    ],
    # o tempo fora: 3 h, 3h, 3,0 horas, tres horas, 3 hours, ~180 min
    "tempo": [
        r"\b3(?:\.0+)?\s*h(?:ora|our)?s?\b",
        r"tres\s+horas?\b",
        r"\b1[78]\d\s*min",
    ],
}


def detalhar_evidencia(resposta: str) -> dict:
    """Diz, fato por fato, se a resposta o cita. Use para depurar o avaliador."""
    t = _normalizar(resposta or "")
    return {nome: any(re.search(p, t) for p in padroes)
            for nome, padroes in FATOS.items()}


def avaliar_evidencia(resposta: str) -> int:
    """Nota de 0 a 3: quantos dos tres fatos a resposta final cita.

    NAO avalia se a resposta esta correta — avalia se ela e auditavel.
    """
    return sum(detalhar_evidencia(resposta).values())


# Bateria do proprio avaliador. As DEZ "ricas" dizem A MESMA COISA em
# formatos diferentes — se alguma nao tirar 3, o avaliador esta punindo
# ESTILO em vez de medir CONTEUDO, e a hipotese H1 fica impossivel de
# interpretar. As cinco "pobres" existem para pegar falso positivo.
_pobres = [
    ("O lote L-77 nao pode ser usado.", 0),
    ("Nao recomendo o uso; a temperatura esta fora do especificado.", 0),
    ("A camara marca 9,8 C e esta fora da faixa. Nao use.", 1),
    ("CF-02: 2 excursoes nas ultimas 24h. Pico de 8,6 C por 40 min.", 0),
    ("Faixa de 2 a 8 C. Nao ha leitura disponivel.", 1),
]
_ricas = [
    "Nao use o L-77: a camara CF-02 marca 9,8 C contra a faixa permitida "
    "de 2 a 8 C, e esta fora ha cerca de 3 horas.",
    "Leitura de 9.8 graus, faixa 2-8 graus, fora da faixa por 3h.",
    "A camara esta a 9,8 C, entre 2 e 8 C e o permitido, e ja sao 3,0 horas fora.",
    "9,8 C medidos; especificacao 2 C a 8 C; excursao iniciada ha 3 horas.",
    # a que quebrou o avaliador na execucao real de 24/08:
    "Faixa permitida: 2 °C a 8 °C. Atual: 9,8 °C. Fora ha aproximadamente 3 horas.",
    "Faixa 2°C–8°C, leitura 9,8°C, ~3h fora.",
    "Permitido de 2 ate 8 graus Celsius; medido 9,8; tempo fora 3 h.",
    "range 2 to 8 C, reading 9.8 C, out for 3 hours",
    "A faixa vai de 2 a 8. Leitura: 9,8. Fora ha 180 min.",
]
for _t, _esperado in _pobres:
    _n = avaliar_evidencia(_t)
    print(f"  pobre: {_n}/3  {detalhar_evidencia(_t)}")
    assert _n == _esperado, f"FALSO POSITIVO em: {_t}"
for _t in _ricas:
    _n = avaliar_evidencia(_t)
    print(f"  rica : {_n}/3  {detalhar_evidencia(_t)}")
    assert _n == 3, f"FALSO NEGATIVO em: {_t}"
print()
print(f"avaliador conferido: {len(_pobres)} pobres e {len(_ricas)} ricas.")
print("As ricas dizem o MESMO conteudo em formatos diferentes, e todas")
print("tiram 3. Rubrica que pune estilo nao mede conteudo.")

  pobre: 0/3  {'leitura': False, 'faixa': False, 'tempo': False}
  pobre: 0/3  {'leitura': False, 'faixa': False, 'tempo': False}
  pobre: 1/3  {'leitura': True, 'faixa': False, 'tempo': False}
  pobre: 0/3  {'leitura': False, 'faixa': False, 'tempo': False}
  pobre: 1/3  {'leitura': False, 'faixa': True, 'tempo': False}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}
  rica : 3/3  {'leitura': True, 'faixa': True, 'tempo': True}

avaliador conferido: 5 pobres e 9 ricas.
As ricas dizem o MESMO conteudo em formatos diferentes, e todas
tiram 3. Rubr

## Parte 6 — A função de medir

Uma execução do agente já entrega **os quatro indicadores de uma vez**: o retorno diz se concluiu e quantas chamadas se perderam, a resposta final dá a nota de evidência, e o `usage` dá o custo. Então medir é rodar `n` vezes e resumir.

**`n = 2` por padrão, e o motivo é o limite de taxa.** A conta mudou em relação ao Encontro 3, e vale fazê-la em voz alta:

| | Encontro 3 (canal de texto) | Hoje (canal nativo) |
|---|---|---|
| voltas por execução | 3 | **3 ou 4** |
| a declaração de ferramentas... | ...ia **uma vez**, na instrução | ...vai **em toda volta**, no `tools` |
| tokens por execução | ~2.000 | **~3.300** *(estimado)* |
| execuções que cabem em 8.000 TPM | 4 | **2** |

> **Uma medição por minuto**, e agora com duas execuções em vez de três. A amostra que você perde aqui, você recupera rodando a medição **duas vezes** — que é o que o Lab 1 pede de qualquer jeito, para separar efeito de variância.

E o número `~3.300` é **estimativa**, não medição: sai da soma do que o Encontro 3 mediu com o tamanho da declaração de `tools`. **A célula de calibração do Lab 0 mede o valor real.** Se discordarem, o material está errado e a calibração está certa.

In [9]:
def medir(instrucao: str, n: int = 2, rotulo: str = "", tools=None,
          verboso: bool = True) -> dict:
    """Roda o agente n vezes e resume os quatro indicadores.

    Args:
        instrucao: a instrucao de sistema a testar
        n: quantas execucoes. 2 por padrao, e o motivo e o TPM de 8.000:
           no canal nativo a declaracao de tools e reenviada em TODA
           volta, e uma execucao passou a custar ~3.300 tokens. Duas
           cabem no minuto; tres nao. A celula de calibracao do Lab 0
           mede o seu caso e confere este numero.
        rotulo: nome da variante, so para a impressao
        tools: declaracao a usar. None = TOOLS (a rica).
    """
    notas, ent, sai = [], 0, 0
    concluidas = chamadas = ruins = voltas = 0
    modelo_no_inicio = LLM_MODEL

    for i in range(1, n + 1):
        r = agente(PERGUNTA, instrucao, tools=tools)
        nota = avaliar_evidencia(r["resposta"])
        notas.append(nota)
        concluidas += r["concluiu"]
        chamadas += r["chamadas"]
        ruins += r["ruins"]
        voltas += r["voltas"]
        ent += r["entrada"]
        sai += r["saida"]
        if verboso:
            print(f"  exec {i}: evidencia {nota}/3 | "
                  f"{'concluiu' if r['concluiu'] else 'ESTOUROU O ORCAMENTO'} | "
                  f"{r['voltas']} voltas | {r['chamadas']} chamadas "
                  f"({r['ruins']} mal aproveitadas) | "
                  f"{r['entrada']}+{r['saida']} tokens")

    total = custo_usd(ent, sai)
    aceitaveis = sum(1 for x in notas if x >= 2)
    assert LLM_MODEL == modelo_no_inicio, (
        "o modelo mudou no meio da medicao — a amostra nao e comparavel")
    res = {
        "rotulo": rotulo, "n": n, "modelo": LLM_MODEL,
        "evidencia_media": sum(notas) / n,
        "concluidas": concluidas,
        "chamadas": chamadas, "ruins": ruins, "voltas": voltas,
        "entrada": ent, "saida": sai,
        "custo_total": total,
        "aceitaveis": aceitaveis,
        # A METRICA QUE DECIDE. Se nenhuma resposta foi aceitavel, o custo por
        # resposta aceitavel e INFINITO — e essa e a resposta honesta.
        # Nao zero, nao None: infinito. Voce gastou e nao levou nada.
        "custo_por_aceitavel": (total / aceitaveis) if aceitaveis else float("inf"),
    }
    if verboso:
        print(f"\n  [{rotulo}] modelo {LLM_MODEL}")
        print(f"  evidencia media {res['evidencia_media']:.2f}/3 | "
              f"concluiu {concluidas}/{n} | aceitaveis {aceitaveis}/{n}")
        print(f"  voltas {voltas} | chamadas {chamadas} "
              f"({ruins} mal aproveitadas) | US$ {total:.6f}")
        print(f"  saida foi {sai / (ent + sai) * 100:.0f}% dos tokens e "
              f"{custo_usd(0, sai) / total * 100:.0f}% do custo")
    return res


print("funcao de medir pronta — quatro indicadores, mais voltas e modelo.")

funcao de medir pronta — quatro indicadores, mais voltas e modelo.


## Parte 7 — Lab 0: a linha de base

**Rode isto antes de qualquer discussão.** É o "antes" do experimento, e tem de ser medido **sem nenhum ensinamento pelo meio** — senão não é linha de base, é resultado.

São duas células, e a primeira faz um trabalho que o Encontro 3 não precisava fazer:

1. **calibração** — uma execução com rastro, que mede **quanto custa uma execução no canal nativo** e diz quantas cabem no seu minuto;
2. **a linha de base** — duas execuções, depois de um minuto de espera.

> **Por que duas execuções e não três.** No canal nativo a declaração de `tools` é **reenviada em toda volta** do laço, e o número de voltas subiu — três ferramentas pedidas uma por vez, mais a volta da resposta. A estimativa é de **~3.300 tokens por execução**, contra ~2.000 no Encontro 3. Com três execuções você pede ~9.900 tokens num minuto, e o teto é 8.000.
>
> **A célula de calibração confere isso no seu caso.** Se a conta dela discordar da estimativa, **ela está certa e eu estou errado** — anote o número e traga.

Depois de rodar, leve **quatro** números ao quadro: conclusão, chamadas mal aproveitadas, evidência média e custo.

In [10]:
# CALIBRACAO. Uma execucao com rastro, que serve a dois propositos:
# ver o canal funcionando, e MEDIR quanto custa uma execucao no seu caso.
print("=" * 66)
print("UMA execucao com rastro — olhe as chamadas e o papel tool")
print("=" * 66)
_demo = agente(PERGUNTA, INSTRUCAO_BASE, verboso=True)

_tokens = _demo["entrada"] + _demo["saida"]
TPM = 8_000
print()
print("-" * 66)
print(f"evidencia desta execucao : {avaliar_evidencia(_demo['resposta'])}/3")
print(f"voltas                   : {_demo['voltas']}")
print(f"tokens                   : {_demo['entrada']} entrada + "
      f"{_demo['saida']} saida = {_tokens}")
print(f"custo                    : US$ {custo_usd(_demo['entrada'], _demo['saida']):.6f}")
print()
print(f"O seu TPM e {TPM:,}. Uma execucao usa {_tokens / TPM * 100:.0f}% dele.")
print(f"Cabem {TPM // _tokens} execucoes por minuto.")
if TPM // _tokens < 2:
    print(">> ATENCAO: nem duas cabem. Rode as medicoes com n=1 e compense")
    print("   fazendo QUATRO medicoes em vez de duas.")
else:
    print(">> n=2 cabe. E o padrao de medir().")
print()
print("Compare com o Encontro 3: la uma execucao custava ~2.000 tokens, em")
print("3 voltas. O que cresceu foi a ENTRADA — a declaracao de tools e")
print("reenviada em toda volta. Trocar o canal nao foi de graca.")

UMA execucao com rastro — olhe as chamadas e o papel tool
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 4 ----------------------------------------------
RESPOSTA FINAL: **Resposta ao responsável pelo almoxarifado**

O lote **L‑77** está armazenado na câmara CF‑02, cuja faixa de temperatura permitida é de **2 °C a 8 °C**. A validade do lote é 30 de novembro de 2026, portanto não há problema de prazo.

**Situação atual da câmara CF‑02**

| Item

In [11]:
# ESPERE UM MINUTO. A celula anterior ja consumiu parte do seu TPM,
# e a linha de base precisa do minuto inteiro para nao pegar 429.
import time; time.sleep(60)

base = medir(INSTRUCAO_BASE, n=2, rotulo="base")

  exec 1: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+1273 tokens
  exec 2: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+997 tokens

  [base] modelo openai/gpt-oss-20b
  evidencia media 3.00/3 | concluiu 2/2 | aceitaveis 2/2
  voltas 8 | chamadas 6 (0 mal aproveitadas) | US$ 0.001057
  saida foi 31% dos tokens e 64% do custo


## Parte 8 — Lab 1: a sua hipótese

As quatro variantes **já estão escritas** na célula abaixo: você troca o rótulo `HIPOTESE` e roda. Mude **só isso** — mexendo em duas coisas você não saberá a qual atribuir o efeito.

| # | O que muda | O que se espera mover | A pergunta de fundo |
|---|---|---|---|
| **H1** | a instrução pede a evidência | **evidência** sobe; saída provavelmente sobe também | dá para exigir **conteúdo** da resposta pelo texto? |
| **H2** | as **descrições** ficam pobres, instrução intacta | chamadas mal aproveitadas **sobem**; voltas sobem | quanto vale uma boa *docstring*, **em número**? |
| **H3** | a instrução perde as **regras de domínio** | conclusão e evidência podem cair | o procedimento mora **na instrução** ou **nas ferramentas**? |
| **H4** | `20b` → `120b`, nada mais | custo **dobra** | a qualidade acompanha o preço? |

### H2 e H3 são um par, e é o par que ensina

As duas atacam o **mesmo comportamento** por lados opostos. A H2 degrada as **descrições** e mantém a instrução; a H3 degrada a **instrução** e mantém as descrições. Juntas, elas respondem a uma pergunta que nenhuma responde sozinha:

> **Quando o agente acerta o procedimento — consultar a ficha antes de medir — quem está mandando: o texto da instrução ou o texto das descrições?**

A resposta muda o que você escreve no projeto da sua equipe. Se o comportamento vive nas descrições, instrução longa é desperdício. Se vive na instrução, *docstring* enxuta basta. **Não decida por gosto: as duas equipes trazem os números e a turma compara.**

> **Rode a medição DUAS vezes**, com um minuto entre elas. Cada medição tem `n = 2`, então você termina com **quatro execuções** da sua variante. A diferença entre as duas medições é a **variância** — e se ela for do tamanho do seu efeito, você não demonstrou nada. Esse também é um resultado, e é um resultado honesto.
>
> **Por que não uma medição de `n = 4`:** porque quatro execuções não cabem no mesmo minuto. A restrição de taxa virou decisão de desenho experimental — e isso não é acidente do laboratório, é como funciona medir com orçamento.

In [12]:
# As quatro variantes, JA ESCRITAS. Cada uma muda UMA coisa.

# ---------- H1: a instrucao pede a evidencia ----------
INSTRUCAO_H1 = INSTRUCAO_BASE.replace(_FECHO, _FECHO + """
Cite SEMPRE tres coisas nessa resposta final: a leitura em graus Celsius, a
faixa permitida do lote, e ha quanto tempo a camara esta fora da faixa.
Sem esses tres numeros a resposta nao serve para auditoria.""")
TOOLS_H1 = TOOLS

# ---------- H2: descricoes pobres, instrucao intacta ----------
INSTRUCAO_H2 = INSTRUCAO_BASE
TOOLS_H2 = TOOLS_POBRE

# ---------- H3: instrucao sem as regras de dominio ----------
# Saiu "comece pela especificacao" e saiu "verifique o historico".
# O que sobrou: quem ele e, e que nao deve inventar dado.
INSTRUCAO_H3 = """Voce e um agente que avalia se insumos refrigerados podem ser usados.

Use as ferramentas disponiveis para obter os dados. Nunca invente uma leitura
nem uma faixa de temperatura: se precisa de um dado, chame a ferramenta.

Quando tiver todos os dados, responda em texto ao responsavel pelo almoxarifado.
"""
TOOLS_H3 = TOOLS

# ---------- H4: nada muda no texto, so o modelo ----------
INSTRUCAO_H4 = INSTRUCAO_BASE
TOOLS_H4 = TOOLS


# Conferencia das variantes, para voce confiar no que vai medir.
assert "auditoria" in INSTRUCAO_H1 and "auditoria" not in INSTRUCAO_BASE
assert "comece pela especificacao" in INSTRUCAO_BASE
assert "comece pela especificacao" not in INSTRUCAO_H3
assert "historico de excursoes" not in INSTRUCAO_H3
assert len(json.dumps(TOOLS_POBRE)) < len(json.dumps(TOOLS))

# =====================================================================
# ESCOLHA A SUA: troque o rotulo abaixo e rode.
# =====================================================================
HIPOTESE = "H4"

_VARIANTES = {
    "H1": (INSTRUCAO_H1, TOOLS_H1),
    "H2": (INSTRUCAO_H2, TOOLS_H2),
    "H3": (INSTRUCAO_H3, TOOLS_H3),
    "H4": (INSTRUCAO_H4, TOOLS_H4),
}
INSTRUCAO_VARIANTE, TOOLS_VARIANTE = _VARIANTES[HIPOTESE]

# H4 e a unica que troca o modelo. As outras ficam no 20b.
if HIPOTESE == "H4":
    LLM_MODEL = "openai/gpt-oss-120b"
    print("H4: modelo trocado para", LLM_MODEL)

print(f"hipotese {HIPOTESE}")
print(f"  instrucao: {len(INSTRUCAO_VARIANTE):5d} caracteres "
      f"(base tem {len(INSTRUCAO_BASE)})")
print(f"  tools    : {len(json.dumps(TOOLS_VARIANTE)):5d} caracteres "
      f"(rica tem {len(json.dumps(TOOLS))})")
print(f"  modelo   : {LLM_MODEL}")
print()
mudou_txt = INSTRUCAO_VARIANTE != INSTRUCAO_BASE
mudou_tls = TOOLS_VARIANTE is not TOOLS
print(f"mudou a instrucao? {mudou_txt}   mudou as descricoes? {mudou_tls}")
if mudou_txt and mudou_tls:
    print("ATENCAO: duas variaveis mudaram. O experimento nao e atribuivel.")

H4: modelo trocado para openai/gpt-oss-120b
hipotese H4
  instrucao:   593 caracteres (base tem 593)
  tools    :  1469 caracteres (rica tem 1469)
  modelo   : openai/gpt-oss-120b

mudou a instrucao? False   mudou as descricoes? False


In [13]:
# ESPERE UM MINUTO. A linha de base acabou de consumir ~83% do seu TPM.
# Sem esta espera voce pede ~13.200 tokens no mesmo minuto, contra teto 8.000.
import time; time.sleep(60)

v1 = medir(INSTRUCAO_VARIANTE, n=2, tools=TOOLS_VARIANTE,
           rotulo=f"{HIPOTESE} - medicao 1")

  exec 1: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+745 tokens
  exec 2: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+734 tokens

  [H4 - medicao 1] modelo openai/gpt-oss-120b
  evidencia media 3.00/3 | concluiu 2/2 | aceitaveis 2/2
  voltas 8 | chamadas 6 (0 mal aproveitadas) | US$ 0.001640
  saida foi 23% dos tokens e 54% do custo


In [14]:
# ESPERE UM MINUTO antes de rodar esta celula. TPM de 8.000.
import time; time.sleep(60)

v2 = medir(INSTRUCAO_VARIANTE, n=2, tools=TOOLS_VARIANTE,
           rotulo=f"{HIPOTESE} - medicao 2")

  exec 1: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+788 tokens
  exec 2: evidencia 3/3 | concluiu | 4 voltas | 3 chamadas (0 mal aproveitadas) | 2509+794 tokens

  [H4 - medicao 2] modelo openai/gpt-oss-120b
  evidencia media 3.00/3 | concluiu 2/2 | aceitaveis 2/2
  voltas 8 | chamadas 6 (0 mal aproveitadas) | US$ 0.001702
  saida foi 24% dos tokens e 56% do custo


In [15]:
# Cada hipotese prediz que um indicador DIFERENTE vai se mover. O veredito
# estatistico tem de ser calculado sobre ESSE indicador, nao sobre a evidencia
# de todo mundo — senao tres das quatro equipes recebem um julgamento sobre
# uma metrica que nao e a delas.
INDICADOR_DA_HIPOTESE = {
    "H1": ("evidencia_media", "evidencia media (0 a 3)", 1),
    "H2": ("ruins",           "chamadas mal aproveitadas", -1),
    "H3": ("concluidas",      "execucoes que concluiram", 1),
    "H4": ("evidencia_media", "evidencia media (0 a 3)", 1),
}


def comparar(base, a, b, hipotese=None):
    """Mostra o efeito da variante ao lado da variacao entre MEDICOES.

    Nao confunda: dentro de uma medicao ha n execucoes, e a media delas e UM
    ponto. A variacao que interessa e entre os DOIS pontos.

    Args:
        base: resultado do Lab 0
        a, b: as duas medicoes da variante
        hipotese: rotulo. Se omitido, usa a variavel global HIPOTESE.
    """
    hip = hipotese or HIPOTESE
    chave, nome_do_indicador, sentido = INDICADOR_DA_HIPOTESE[hip]

    def por_exec(r, k):
        """Normaliza por execucao — as medicoes podem ter n diferente."""
        return r[k] / r["n"]

    print(f"{'':24} {'base':>10} {'variante':>10}")
    print("-" * 48)
    for k, rot in (("evidencia_media", "evidencia media"),
                   ("concluidas", "concluiu (por exec)"),
                   ("voltas", "voltas (por exec)"),
                   ("chamadas", "chamadas (por exec)"),
                   ("ruins", "MAL APROVEITADAS")):
        vb = base[k] if k == "evidencia_media" else por_exec(base, k)
        va = ((a[k] + b[k]) / 2 if k == "evidencia_media"
              else (por_exec(a, k) + por_exec(b, k)) / 2)
        marca = "  <<<" if k == chave else ""
        print(f"{rot:24} {vb:>10.2f} {va:>10.2f}{marca}")
    cb = base["custo_total"] / base["n"]
    cv = (a["custo_total"] + b["custo_total"]) / (a["n"] + b["n"])
    print(f"{'US$ por execucao':24} {cb:>10.6f} {cv:>10.6f}")
    print(f"{'modelo':24} {base['modelo']:>10} {a['modelo']:>10}")

    # O veredito, sobre o indicador DA HIPOTESE.
    if chave == "evidencia_media":
        vb = base[chave]; xa = a[chave]; xb = b[chave]
    else:
        vb = por_exec(base, chave); xa = por_exec(a, chave); xb = por_exec(b, chave)
    efeito = (xa + xb) / 2 - vb
    ruido = abs(xa - xb)

    print()
    print(f"HIPOTESE {hip} — o indicador que ela prediz mover:")
    print(f"  {nome_do_indicador}")
    print(f"  base {vb:.2f}  ->  variante {(xa + xb) / 2:.2f}")
    print(f"  EFEITO                 : {efeito:+.2f}"
          f"   ({'na direcao prevista' if efeito * sentido > 0 else 'CONTRA a previsao' if efeito else 'nulo'})")
    print(f"  VARIACAO entre medicoes: {ruido:.2f}")
    print()
    if ruido >= abs(efeito):
        print("  >> A variacao e do tamanho do efeito. VOCE NAO DEMONSTROU NADA.")
        print("     Isso e um resultado valido, e a conclusao e metodologica:")
        print("     com n=2 e duas medicoes nao da para afirmar.")
        print("     Quantas amostras seriam precisas? (Encontro 13)")
    else:
        print("  >> O efeito e maior que a variacao. Ha indicio, com 4 execucoes.")
    print()


comparar(base, v1, v2)

                               base   variante
------------------------------------------------
evidencia media                3.00       3.00  <<<
concluiu (por exec)            1.00       1.00
voltas (por exec)              4.00       4.00
chamadas (por exec)            3.00       3.00
MAL APROVEITADAS               0.00       0.00
US$ por execucao           0.000529   0.000835
modelo                   openai/gpt-oss-20b openai/gpt-oss-120b

HIPOTESE H4 — o indicador que ela prediz mover:
  evidencia media (0 a 3)
  base 3.00  ->  variante 3.00
  EFEITO                 : +0.00   (nulo)
  VARIACAO entre medicoes: 0.00

  >> A variacao e do tamanho do efeito. VOCE NAO DEMONSTROU NADA.
     Isso e um resultado valido, e a conclusao e metodologica:
     com n=2 e duas medicoes nao da para afirmar.
     Quantas amostras seriam precisas? (Encontro 13)



## Parte 9 — Quando a garantia não cabe no agente

Até aqui vocês **pediram** coisas e mediram se o modelo obedeceu. A pergunta agora muda: **existe forma de não depender de ele obedecer?**

Existe, e ela tem nome: `response_format` com `strict: true`. A documentação do Groq é literal — *"never errors or produces invalid JSON"*, *"100% schema adherence"*. Não é validação depois: é **decodificação restrita**, o modelo fica impedido de emitir um token que quebre o esquema. Os dois modelos da disciplina suportam.

**E ela não funciona junto com ferramentas.** A mesma documentação, a mesma página:

> *"Streaming and **tool use** are not currently supported with Structured Outputs."*

Então a escada de hoje tem **dois degraus reais**, não três, e a diferença entre eles é de **arquitetura**:

| Caminho | Como | Garantia | Custo |
|---|---|---|---|
| **L1** · pedir e reparar | uma chamada com `tools`, JSON pedido no texto, validado em Python, reparado por realimentação | **nenhuma** — depende de obediência | 1 laço + reparos |
| **L2** · duas chamadas | o agente age com `tools`; depois **uma segunda chamada, sem `tools`**, converte a resposta em JSON com `strict: true` | **total, na segunda** | 1 laço + 1 chamada |

**O L2 é o padrão que se usa de verdade em produção**, e o nome dele é separação de responsabilidades: uma chamada **age**, outra **formata**. Cada uma recebe a garantia que o protocolo sabe dar.

**Duas restrições do modo estrito**, e a segunda é decisão de projeto disfarçada de detalhe:

- `additionalProperties: false` é obrigatório;
- **todo campo tem de ser `required`.** Não existe campo opcional. Se a excursão pode não ter ocorrido, você **não omite o campo** — declara uma sentinela, tipo `"nenhuma"`.

> **Quem quer garantia paga em rigidez.** Anotem a frase; ela volta no Encontro 16, com atuador saturado no lugar de esquema JSON.

### L1 — pedir JSON no texto, e reparar quando vier errado

Garantia zero: você **pede** JSON na instrução, valida em Python e, quando falha, **devolve a mensagem de erro ao modelo** e pede de novo.

Isso tem nome — **reparo por realimentação** — e é a técnica mais útil do nível 1. O detalhe que a faz funcionar: não basta dizer *"errado, tente de novo"*. Você tem de devolver **o erro específico**, porque é ele que diz ao modelo o que consertar.

Repare que o L1 **mantém o agente inteiro**: o laço com `tools` roda normalmente, e só a resposta final é que precisa ser JSON.

### L2 — duas chamadas: uma age, outra formata

Aqui está a arquitetura que o conflito nos obriga a adotar, e que é a que se usa de verdade:

```
chamada 1..n   agente com tools  →  resposta em TEXTO   (garantia de esquema: nenhuma)
chamada n+1    formatador sem tools, strict: true  →  JSON  (garantia: total)
```

O formatador **não tem ferramentas e não decide nada**: ele recebe o texto que o agente produziu e o converte. É deliberadamente burro, e é por isso que dá para garanti-lo.

**O preço é uma chamada extra**, e ela não é grátis: entra o texto da resposta mais o esquema, sai o JSON. A tabela da Parte 10 vai dizer se valeu.

> **A pergunta desconfortável, e ela é boa:** se o formatador só transcreve, ele pode **perder** informação que estava no texto? Pode. É por isso que rodamos o avaliador de evidência **no JSON final**, não no texto intermediário.

## Parte 10 — Caminho estendido (opcional)

Os dois primeiros são os de maior retorno, e o primeiro mexe direto na conta que a aula inteira mediu.

1. **`reasoning_effort`, e é o item mais valioso da lista.** Os dois `gpt-oss` aceitam `reasoning_effort="low"`, `"medium"` ou `"high"`, que controla quantos tokens de raciocínio o modelo gasta. Raciocínio é cobrado como **saída**, e saída é a metade do custo. Rode a linha de base com `low` e com `high` — `chamar(..., reasoning_effort="low")`, que o `**extra` já repassa — e meça **as três coisas**: custo, evidência e chamadas mal aproveitadas.
   > **Faça a previsão antes de rodar.** `low` deve sair mais barato; a pergunta é se ele ainda encadeia as três ferramentas na ordem certa. Se sair mais barato **e** manter a evidência, vocês acabaram de achar a economia que nenhuma hipótese da aula achou.

2. **O par H2 × H3, feito inteiro:** rode as duas e responda **com números** onde mora o procedimento — na instrução ou nas descrições. É o desenho mais forte do encontro e nenhuma equipe consegue fazê-lo sozinha.

3. **A recomendação do provedor que nós contrariamos.** A documentação de *reasoning* do Groq diz, para modelos de raciocínio: *"avoid system prompts — include all instructions in the user message"*. **Nós pusemos tudo no `system`.** Mova a instrução para a mensagem `user` e meça. Se der diferença, o material está errado e vocês descobriram. Se não der, vocês mostraram que a recomendação não vale para este caso — e as duas conclusões valem a mesma coisa.

4. **Rode a medição cinco vezes** em vez de duas, com um minuto entre cada, e estime o desvio de verdade. É o Encontro 13 antecipado, e pode mudar a leitura de toda a tabela do quadro. Custa ~17 mil tokens, dentro do seu dia.

5. **Teste a hipótese de outra equipe** e veja se o resultado se reproduz. Reprodução independente vale mais que efeito grande.

6. **Force o erro de argumento:** tire o `"Exemplo: L-77"` da `description` do `lote`, deixando o resto intacto, e veja se o modelo passa a chutar o formato. É a linha 3 da clínica, medida — e é a variante de **uma palavra**, a mais limpa de todas.

7. **Ferramenta em paralelo:** faça uma pergunta que precise de dois dados independentes — duas câmaras, por exemplo — e veja se o modelo pede as duas na mesma volta. Conte as voltas economizadas e converta em dólares.

8. **Resolva o campo opcional:** num esquema em que todo campo é obrigatório, como representar *"não houve excursão"*? Compare a sentinela `"nenhuma"` com um booleano extra. Qual das duas o modelo preenche melhor?

9. **Melhore o avaliador:** hoje ele procura três números. Faça-o distinguir *"citou o valor"* de *"citou o valor certo"* — e note que isso o aproxima de verificar **correção**, não só auditabilidade. É o Encontro 13.

In [16]:
# Espaco livre para o caminho estendido.

## Parte 11 — Salvar e entregar

**Arquivo → Salvar uma cópia no GitHub**, em `notebooks/enc04_<seu-nome>.ipynb`, com a mensagem `encontro 4: canal nativo, hipotese <H?> medida`.

### Verificação final

- [ ] o notebook roda de ponta a ponta em **sessão limpa**
- [ ] a hipótese escolhida está declarada em `HIPOTESE`, e é **uma só**
- [ ] a linha de base do Lab 0 está registrada, com `n` e **modelo**
- [ ] o **indicador da sua hipótese** está identificado, e o efeito e a variação são **dele** — não da evidência, se a sua hipótese não é H1 nem H4
- [ ] a hipótese foi medida **duas vezes**, e a **variação entre medições** está anotada ao lado do efeito
- [ ] o número que a **calibração** do Lab 0 imprimiu está anotado, com o `n` que ele permitiu
- [ ] o L1 e o L2 rodaram, e está anotado **quantas vezes cada um falhou**
- [ ] o **custo por resposta aceitável** está calculado nos dois caminhos
- [ ] a mensagem de erro do conflito `tools` + `strict` está **salva na saída da célula**
- [ ] nenhuma chave aparece em nenhuma célula

### Para o repositório da equipe

| Arquivo | O que vai nele |
|---|---|
| `docs/ferramentas.md` | as três descrições **em formato `tools`**, com o esquema dos argumentos |
| `docs/instrucao.md` | a variante que venceu, **uma linha por decisão** |
| `docs/medicoes.md` | os quatro indicadores **mais voltas**, com `n`, **modelo** e a variação entre **medições** |
| `docs/custo.md` | custo por resposta aceitável do L1 e do L2 |
| `evals/avaliar_resposta.py` | o avaliador da Parte 5, copiado inteiro: `_normalizar`, `FATOS`, `detalhar_evidencia`, `avaliar_evidencia` **e os testes**. O arquivo se chama `avaliar_resposta.py` e a função `avaliar_evidencia` — o arquivo é o módulo, a função é uma das que ele vai ter |
| `docs/aceitacao.md` | **uma frase por integrante**: *"para o nosso problema, uma resposta boa precisa citar ___"* |

> A `docs/ferramentas.md` mudou de natureza hoje. Antes era prosa; agora é **declaração com esquema de argumento** — quase código. É o artefato que o Encontro 5 vai estressar.
>
> E a `docs/aceitacao.md` continua sendo a mais curta e a mais importante. **Decidir o que uma resposta boa precisa conter é decidir o critério de aceitação do projeto** — e é de onde o Encontro 13 vai partir.